#Part 1: Python & Data Foundations (Inget, Validate, Clean, Enrich)
###
Task 1.1: Read via csv module

In [92]:
import csv
import json

accepted=[]
rejected=[]

def parse_row(row):
  row['age']=int(row['age'])
  row['fee_inr']=int(row['fee_inr'])
  row['prior_no_shows']=int(row['prior_no_shows'])
  row['distance_km']=float(row['distance_km'])
  return row

with open("/content/appointments.csv",'r',newline='') as file:
  reader=csv.DictReader(file)

  for row in reader:
    try:
      parse_row(row)
      accepted.append(row)
    except ValueError as e:
      rejected.append(row)
print(f"{len(accepted)} accepted rows")
print(f"{len(rejected)} rejected rows")

with open("/content/rejected_rows.json",'w')as file:
  json.dump(rejected,file,indent=2)

988 accepted rows
32 rejected rows


####Task 1.2: Inspect and Clean

In [102]:
import pandas as pd

df=pd.read_csv("/content/appointments.csv")

print(f"Shape of the data :{df.shape}")
print(f"info of the data :{df.info()}")
print(f"Number of null values :{df.isnull().sum()}")
print(f"Duplicate data :{df.duplicated().sum()}")

df.drop_duplicates(inplace=True)

df['booking_channel']=df['booking_channel'].str.strip().str.title()
df['appt_date']=pd.to_datetime(df['appt_date'],errors='coerce')
df['booked_on']=pd.to_datetime(df['booked_on'],errors='coerce')
df.dropna(subset=['appt_date','status'])

df.shape

Shape of the data :(1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
info of the data :None
Number of null values :appointment_id  

(1000, 14)

###Task 1.3: Clean

In [103]:

df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].astype(int)

df = df[(df['age'] > 1) & (df['age'] < 100)]

df['distance_km'] = df['distance_km'].fillna(
    df['distance_km'].median()
)

q1_distance = df['distance_km'].quantile(0.25)
q3_distance = df['distance_km'].quantile(0.75)

iqr_distance = q3_distance - q1_distance

lower_distance = q1_distance - 1.5 * iqr_distance
upper_distance = q3_distance + 1.5 * iqr_distance

df = df[
    (df['distance_km'] >= lower_distance) &
    (df['distance_km'] <= upper_distance)
]

q1_fee_inr = df['fee_inr'].quantile(0.25)
q3_fee_inr = df['fee_inr'].quantile(0.75)

iqr_fee_inr = q3_fee_inr - q1_fee_inr

lower_fee_inr = q1_fee_inr - 1.5 * iqr_fee_inr
upper_fee_inr = q3_fee_inr + 1.5 * iqr_fee_inr

df = df[
    (df['fee_inr'] >= lower_fee_inr) &
    (df['fee_inr'] <= upper_fee_inr)
]

df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round(2)

df.shape


(966, 14)

###Task 1.4:Features & NumPy Summary

In [104]:
import numpy as np

df['lead_days'] = df['appt_date'] - df['booked_on']
df['lead_days'] = df['lead_days'].dt.days
df['no_show'] = df['status'].map({'No-Show': 1, 'Attended': 0})

x = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Column means (axis=0)
col_means = np.mean(x, axis=0)
print("Column means:", col_means)

# (b) Lead days > 14 count and share (using a separate mask, leaving df['lead_days'] as ints)
lead_mask = df['lead_days'] > 14
print(f"lead_days > 14: {lead_mask.sum()} rows ({lead_mask.mean() * 100:.1f}%)")

# (c) Z-scores for distance_km only (column index 1)
dist = x[:, 1]
z_score = (dist - np.mean(dist)) / np.std(dist)
outliers = (np.abs(z_score) > 2).sum()
print(f"Rows with |z| > 2 for distance_km: {outliers}")

Column means: [ 48.08074534   9.01231884 633.79917184          nan]
lead_days > 14: 287 rows (29.7%)
Rows with |z| > 2 for distance_km: 40


###Task 1.5: Enrich Clinics via Open-Meteo API

In [48]:
import requests
import csv

latitude=int(input("Enter the Latitude:"))
longitude=int(input("Enter the Longitude:"))
params={
    "latitude":latitude,
    "longitude":longitude,
    "hourly":"temperature_2m",
    "forecast_days": 1
}
res=requests.get("https://api.open-meteo.com/v1/forecast",params=params,timeout=10)

try:
  if res.status_code==200:
    data=res.json()
    temp_c=data['hourly']['temperature_2m']
    times = data['hourly']['time']
    with open("clinics_enriched.csv",'w')as file:
      writer=csv.DictWriter(file,fieldnames=['temp_c','time'])
      writer.writeheader()
      for temp,time in zip(temp_c,times):
        writer.writerow({'temp_c':temp,'time':time})
  else:
    print(f"Error:{res.status_code}")
except requests.RequestException:
  print(f'Error:{RequestException}')

Enter the Latitude:10
Enter the Longitude:20


###Task 1.6: Export Clean Table & SQL Load Script

In [122]:
df['appointment_id'].sort_values()

print("Null values in the dataset:",df.isnull().sum())
df.dropna(inplace=True)

appointments_clean=df.copy()
appointments_clean.to_csv(
    "appointments_clean.csv",
    index=False
)


import pandas as pd

df = pd.read_csv("appointments_clean.csv")

with open("load_data.sql", "w") as file:

    clinic_ids = df['clinic_id'].unique()

    for clinic_id in clinic_ids:
        query = f"""
INSERT INTO clinics (clinic_id)
VALUES ('{clinic_id}');
"""
        file.write(query)

    for _, row in df.iterrows():

        query = f"""
INSERT INTO appointments (
    appointment_id, patient_id, clinic_id,
    booked_on, appt_date, age, gender,
    distance_km, fee_inr, booking_channel,
    visit_type, reminder_sent, prior_no_shows,
    status, lead_days, no_show
)
VALUES (
    '{row['appointment_id']}',
    '{row['patient_id']}',
    '{row['clinic_id']}',
    '{row['booked_on']}',
    '{row['appt_date']}',
    {row['age']},
    '{row['gender']}',
    {row['distance_km']},
    {row['fee_inr']},
    '{row['booking_channel']}',
    '{row['visit_type']}',
    '{row['reminder_sent']}',
    {row['prior_no_shows']},
    '{row['status']}',
    {row['lead_days']},
    {row['no_show']}
);
"""
        file.write(query)

print("SQL file generated successfully!")


Null values in the dataset: appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
SQL file generated successfully!
